# Étape 2 — Prétraitement & feature engineering

**Objectif** : transformer le dataset brut (68 258 lignes, 18 colonnes) en :
1. `zenith_clean.csv` — transactions nettoyées prêtes pour la modélisation,
2. `zenith_features.csv` — transactions enrichies de toutes les variables dérivées,
3. `product_features.csv` — agrégats par produit (statistiques 36 mois),
4. `preprocessing_report.csv` — trace avant/après de chaque transformation.

**Entrée** : `data/raw/zenith_dataset_brut.csv` + `data/raw/catalogue_produits_250.csv`.

**Sorties** : voir la liste ci-dessus dans `data/processed/`, `data/features/`, `outputs/tables/`.

**Méthodologie** (mémoire §3.3) :
- §3.3.1 Nettoyage — 13 étapes successives, chaque étape produit un delta tracé.
- §3.3.2 Feature engineering — variables temporelles cycliques, financières, rupture.
- §3.3.3 Partitionnement temporel strict (train 2022-08→2024-12, val 2025-Q1, test 2025-04→07).

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', context='talk')

from src.preprocessing import preprocess_pipeline, clean_dataset, engineer_features
from src.utils import load_raw_transactions, load_catalogue, temporal_split, FEATURES_DIR, PROCESSED_DIR

## 1. Exécution du pipeline complet

On lance la fonction `preprocess_pipeline()` qui orchestre toutes les étapes du module `src.preprocessing`.

In [ ]:
clean, features, product_feats, report = preprocess_pipeline()
print(f'Transactions nettoyées : {len(clean):,}'.replace(',', ' '))
print(f'Features par transaction : {features.shape[1]} colonnes')
print(f'Produits agrégés : {len(product_feats)}')
report

## 2. Lecture du rapport étape par étape

Le rapport montre l'évolution du dataset à chaque étape avec les compteurs spécifiques.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(report['etape'], report['lignes'], marker='o', linewidth=2, color='#1f4e79')
ax.fill_between(report['etape'], report['lignes'], alpha=0.2, color='#1f4e79')
ax.set_title("Évolution du nombre de lignes au fil des étapes de prétraitement")
ax.set_ylabel('Nombre de transactions'); ax.tick_params(axis='x', rotation=70)
plt.tight_layout()
plt.savefig(ROOT/'outputs/figures/prep_01_evolution_lignes.png', dpi=120, bbox_inches='tight'); plt.show()

## 3. Inspection des transactions nettoyées

In [ ]:
clean.dtypes

In [ ]:
# Vérification : plus de quantités négatives ni de doublons
print('Quantités négatives :', int((clean['quantite_vendue']<=0).sum()))
print('Doublons exacts    :', int(clean.duplicated().sum()))
print('Dates NaT          :', int(clean['date'].isna().sum()))
print('NaN par colonne :'); clean.isna().sum().sort_values(ascending=False).head(10)

## 4. Features dérivées (échantillon)

In [ ]:
cols_temporelles = ['annee','mois','trimestre','jour_semaine','est_weekend','est_fin_de_mois',
                    'mois_sin','mois_cos','est_rentree_scolaire','est_periode_pic_b2b']
cols_financieres = ['marge_unitaire','benefice_transaction','taux_marge_pct','valeur_stock_immobilisee']
cols_rupture = ['rupture_signalee','jours_consecutifs_rupture']
features[cols_temporelles + cols_financieres + cols_rupture].head(10)

## 5. Distribution des features clés

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
axes[0,0].hist(features['marge_unitaire'].dropna(), bins=60, color='#1f4e79', edgecolor='black')
axes[0,0].set_title('marge_unitaire (USD)'); axes[0,0].set_yscale('log')
axes[0,1].hist(features['taux_marge_pct'].dropna(), bins=40, color='#ff6b6b', edgecolor='black')
axes[0,1].set_title('taux_marge_pct (%)')
axes[1,0].hist(features['valeur_stock_immobilisee'].dropna(), bins=60, color='#6c757d', edgecolor='black')
axes[1,0].set_title('valeur_stock_immobilisee (USD)'); axes[1,0].set_yscale('log')
axes[1,1].hist(features['jours_consecutifs_rupture'].dropna(), bins=40, color='#2d6a4f', edgecolor='black')
axes[1,1].set_title('jours_consecutifs_rupture')
plt.tight_layout()
plt.savefig(ROOT/'outputs/figures/prep_02_features_financieres.png', dpi=120, bbox_inches='tight'); plt.show()

## 6. Agrégats par produit

In [ ]:
product_feats.head(10)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].hist(product_feats['coefficient_variation'].dropna(), bins=40, color='#1f4e79', edgecolor='black')
axes[0].set_title('Distribution du coefficient de variation des ventes mensuelles')
axes[0].axvline(0.5, color='green', ls='--', label='X (CV<0.5)')
axes[0].axvline(1.0, color='orange', ls='--', label='Y (0.5≤CV<1)')
axes[0].legend()
axes[1].scatter(product_feats['jours_depuis_derniere_vente'], product_feats['ca_total_36mois'], alpha=0.6, color='#ff6b6b')
axes[1].set_xlabel('Jours depuis dernière vente'); axes[1].set_ylabel('CA total 36 mois (USD, log)')
axes[1].set_yscale('symlog'); axes[1].set_title('Fraîcheur produit vs CA')
plt.tight_layout()
plt.savefig(ROOT/'outputs/figures/prep_03_agregats_produits.png', dpi=120, bbox_inches='tight'); plt.show()

## 7. Partitionnement temporel

In [ ]:
train, val, test = temporal_split(clean)
split_df = pd.DataFrame({
    'ensemble': ['train','validation','test'],
    'date_min': [train['date'].min().date(), val['date'].min().date(), test['date'].min().date()],
    'date_max': [train['date'].max().date(), val['date'].max().date(), test['date'].max().date()],
    'lignes': [len(train), len(val), len(test)],
    'pct': [round(len(train)/len(clean)*100,1), round(len(val)/len(clean)*100,1), round(len(test)/len(clean)*100,1)],
})
split_df

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
ca = clean.assign(mois=clean['date'].dt.to_period('M').dt.to_timestamp()).groupby('mois')['montant_total'].sum()
ax.plot(ca.index, ca.values, color='#1f4e79', linewidth=2)
ax.axvspan(train['date'].min(), train['date'].max(), alpha=0.15, color='#2d6a4f', label='train')
ax.axvspan(val['date'].min(), val['date'].max(), alpha=0.25, color='#ff9f1c', label='validation')
ax.axvspan(test['date'].min(), test['date'].max(), alpha=0.25, color='#ef476f', label='test')
ax.set_title('Partitionnement temporel — train / validation / test'); ax.set_ylabel('CA (USD)'); ax.legend()
plt.tight_layout()
plt.savefig(ROOT/'outputs/figures/prep_04_partitionnement_temporel.png', dpi=120, bbox_inches='tight'); plt.show()

## 8. Conclusion

- **68 258 → 67 250 transactions** propres (-1.5 %) ; **41 features** dérivées.
- **672 doublons** supprimés, **66 fautes de frappe famille** corrigées, **2 276 prix ×10** divisés par 10, **1 033 prix** imputés par la médiane famille, **336 retours** isolés.
- **1 680 modes de paiement** imputés (B2B → Crédit / B2C → Comptant), **2 531 noms client B2C** mis à Anonyme, **673 stocks** interpolés.
- **Split temporel strict** : train 85 % / val 7 % / test 8 % — pas de fuite de données, l'historique de la validation ne contamine pas le modèle.

Le dataset est désormais prêt pour l'**Étape 3 — Classification ABC × XYZ × K-Means**.
